In [61]:
import tensorflow as tf
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [62]:
data=fetch_california_housing()
x=data.data
y=data.target

In [63]:
df=pd.DataFrame(x,columns=data.feature_names)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


In [64]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
# Keep copies of targets for reference
y_train_orig = y_train.copy()
y_test_orig = y_test.copy()

In [65]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Ensure target variables are defined and unscaled for stability
y_train = y_train_orig
y_test = y_test_orig

In [66]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LeakyReLU

In [67]:
from tensorflow.keras.layers import Dropout

tf.keras.backend.clear_session()
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

In [68]:
# Lower learning rate for more stable convergence
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

In [69]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop training if validation loss stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 1.2856 - mae: 0.7550 - val_loss: 0.5230 - val_mae: 0.5035
Epoch 2/100
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5835 - mae: 0.5430 - val_loss: 0.4444 - val_mae: 0.4697
Epoch 3/100
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5131 - mae: 0.5114 - val_loss: 0.4093 - val_mae: 0.4460
Epoch 4/100
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4674 - mae: 0.4882 - val_loss: 0.3892 - val_mae: 0.4378
Epoch 5/100
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4454 - mae: 0.4731 - val_loss: 0.3764 - val_mae: 0.4321
Epoch 6/100
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4271 - mae: 0.4639 - val_loss: 0.3668 - val_mae: 0.4270
Epoch 7/100
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4126 - mae: 0.4573 - val_loss: 0.3684 - val_mae: 0.4173
Epoch 8/100
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3995 - mae: 0.4491 - val_loss: 0.3508 - val_mae: 0.4139
Epoch 9/100
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/

### Evaluate Model Performance

Now that the model is trained, let's evaluate its performance on the unseen test data.

In [70]:
loss, mae = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test MAE: {mae:.4f}")

Test Loss: 0.2518
Test MAE: 0.3348


In [71]:
from sklearn.metrics import r2_score

y_pred = model.predict(X_test).flatten()
r2 = r2_score(y_test, y_pred)

print("R² Score:", r2)

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 887us/step
R² Score: 0.8078264835764055
